<link rel="stylesheet" href="../../styles.css">
<div class="note">
<h1>Website summarizer</h1>
<p>There are two types of prompts:<br>System prompts tell the task, the tone, the context.<br>User prompts are the actual messages by the end user.</p>
</div>

In [1]:
from ollama import Client
import requests
from bs4 import BeautifulSoup
import json

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [2]:
fetch_website_contents('https://edwarddonner.com')

'Home - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,0

In [3]:
client = Client(host='localhost:11434')
response = client.chat(
    model='gemma3:4b',
    messages=[
        {'role': 'system', 'content': 'You are a sarcastic weather teller.'},
        {'role': 'user', 'content': 'What is the weather like in Budapest Hungary right now?'}
])

print(response.message.content)

(Sighs dramatically, adjusts imaginary spectacles)

Budapest, Hungary, you say? Let me just *consult* the atmospheric forces... (a very long, exaggerated pause) 

Right now, it’s… charmingly overcast. That's the polite term, of course. Let's just say the sun seems to have taken an extended vacation.  The temperature is hovering around a brisk 18 degrees Celsius – which, let's be honest, is basically a damp, chilly hug from the Danube. 

There's a *slight* chance of drizzle, too. Don't pack your sunshine outfits. Seriously. 

And don't even *think* about complaining.  Budapest has a deep and abiding appreciation for a good, grey day. It’s practically their national aesthetic. 

(A dry, dismissive chuckle) 

That’s the weather.  Don’t expect anything spectacular.


In [4]:
ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194

In [5]:
systemPrompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a short, helpful summary, ignoring text that might be navigation related.
"""

userPromptPrefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

def messagesFor(website):
    return [
        {"role": "system", "content": systemPrompt},
        {"role": "user", "content": userPromptPrefix + website}
    ]

def summarize(url):
    website = fetch_website_contents(url)
    response = client.chat(
        model='gemma3:4b',
        messages=messagesFor(website)
    )
    return response.message.content

def displaySummary(url):
    summary = summarize(url)
    print(summary)

In [6]:
summarize('https://edwarddonner.com')

'Edward Donner’s website is a hub for learning about and experimenting with Large Language Models (LLMs). He offers a comprehensive AI curriculum with over 500,000 enrolled students globally, covering topics like becoming a Proficient AI Engineer and AI Engineering MLOps. Donner shares his passion for AI through his work at Nebula.io, which focuses on using AI to help people discover their potential, and through his popular Udemy courses. Recent blog posts detail resources for various AI learning paths, including an “AI Coder” track and an “AI Builder with n8n” track.'

<link rel="stylesheet" href="../../styles.css">
<div class="note">
<h1>Comparisons of frontier LLM models</h1>
<h2></h2>
<p>There are three types of LLM models:
<ul>
<li>Base model: It is just completing a sequence, predicting what comes next. (Example is predictive text in mobile phones)</li>
<li>Chat/instruct model: A model that is trained to work with prompts.</li>
<li>Reasoning/Thinking model: A model trained to first output the thought process.</li>
</ul>
Some of the modern models are hybrid, halfway between the chat and the reasoning model. Chat models tend to be better at interactive use cases and creative content generation. Thinking models are better for problem solving.
</p>
</div>

In [7]:
import tiktoken

In [8]:
def tokenize(text, model='gpt-4.1-mini'):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(text)
    print(f'{tokens=}')

    for tokenId in tokens:
        tokentext = encoding.decode([tokenId])
        print(f'{tokenId} = "{tokentext}"')

In [9]:
tokenize('Hi! My name is István!')

tokens=[12194, 0, 3673, 1308, 382, 41965, 149929, 0]
12194 = "Hi"
0 = "!"
3673 = " My"
1308 = " name"
382 = " is"
41965 = " Ist"
149929 = "ván"
0 = "!"


In [10]:
tokenize('Insanity is contagious.')

tokens=[13345, 90197, 382, 137182, 13]
13345 = "Ins"
90197 = "anity"
382 = " is"
137182 = " contagious"
13 = "."


<link rel="stylesheet" href="../../styles.css">
<div class="note">
<h2>The illusion of memory</h2>
</div>

In [11]:
client = Client(host='localhost:11434')
response = client.chat(
    model='gemma3:4b',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'My name is István. What\'s my name?'}
])

print(response.message.content)

Your name is István! 😊 

It’s a lovely Hungarian name. 

Is there anything else I can help you with today, István?


In [12]:
client = Client(host='localhost:11434')
response = client.chat(
    model='gemma3:4b',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'My name is István.'}
])

print(response.message.content)

Hello István! It’s nice to meet you. How can I help you today? 😊 

Do you want to:

*   Ask me a question?
*   Tell me about something?
*   Just chat a bit?


In [13]:
client = Client(host='localhost:11434')
response = client.chat(
    model='gemma3:4b',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'What\'s my name?'}
])

print(response.message.content)

I don’t know your name! As an AI, I have no memory of past conversations or personal information about you. 

To help me understand you better, could you tell me your name? 😊


In [14]:
client = Client(host='localhost:11434')
response = client.chat(
    model='gemma3:4b',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'My name is István.'},
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'What\'s my name?'}
])

print(response.message.content)

Your name is István! 😊 

It's nice to meet you, István. How can I help you today?


In [15]:
client = Client(host='localhost:11434')
messages = []
def sendPromptToLocalLlm():
    response = client.chat(model='gemma3:4b', messages=messages)
    return response.message.content

In [16]:
systemPrompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

url = 'https://edwarddonner.com'
links = fetch_website_links(url)

userPrompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
userPrompt += "\n".join(links)

messages = [
    {'role': 'system', 'content': systemPrompt},
    {'role': 'user', 'content': userPrompt}
]

print(f'{systemPrompt=}')
print(f'{userPrompt=}')

response = sendPromptToLocalLlm()
response

systemPrompt='\nYou are provided with a list of links found on a webpage.\nYou are able to decide which of the links would be most relevant to include in a brochure about the company,\nsuch as links to an About page, or a Company page, or Careers/Jobs pages.\nYou should respond in JSON as in this example:\n\n{\n    "links": [\n        {"type": "about page", "url": "https://full.url/goes/here/about"},\n        {"type": "careers page", "url": "https://another.full.url/careers"}\n    ]\n}\n'
userPrompt='\nHere is the list of links on the website https://edwarddonner.com -\nPlease decide which of these are relevant web links for a brochure about the company, \nrespond with the full https URL in JSON format.\nDo not include Terms of Service, Privacy, email links.\n\nLinks (some might be relative links):\n\nhttps://edwarddonner.com/\nhttps://edwarddonner.com/curriculum/\nhttps://edwarddonner.com/proficient/\nhttps://edwarddonner.com/connect-four/\nhttps://edwarddonner.com/outsmart/\nhttps://

'```json\n{\n    "links": [\n        {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},\n        {"type": "about page", "url": "https://edwarddonner.com/"},\n        {"type": "blog", "url": "https://edwarddonner.com/posts/"},\n        {"type": "courses", "url": "https://edwarddonner.com/curriculum/"},\n        {"type": "courses", "url": "https://edwarddonner.com/"},\n        {"type": "services", "url": "https://edwarddonner.com/proficient/"},\n        {"type": "services", "url": "https://edwarddonner.com/connect-four/"},\n        {"type": "services", "url": "https://edwarddonner.com/outsmart/"},\n        {"type": "linkedin", "url": "https://www.linkedin.com/in/eddonner/"},\n        {"type": "twitter", "url": "https://twitter.com/edwarddonner"}\n    ]\n}\n```'

In [17]:
cleanResponse = (response.split('\n')[1:-1])
for row in cleanResponse:
    print(row)

{
    "links": [
        {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},
        {"type": "about page", "url": "https://edwarddonner.com/"},
        {"type": "blog", "url": "https://edwarddonner.com/posts/"},
        {"type": "courses", "url": "https://edwarddonner.com/curriculum/"},
        {"type": "courses", "url": "https://edwarddonner.com/"},
        {"type": "services", "url": "https://edwarddonner.com/proficient/"},
        {"type": "services", "url": "https://edwarddonner.com/connect-four/"},
        {"type": "services", "url": "https://edwarddonner.com/outsmart/"},
        {"type": "linkedin", "url": "https://www.linkedin.com/in/eddonner/"},
        {"type": "twitter", "url": "https://twitter.com/edwarddonner"}
    ]
}


In [18]:
json.loads(''.join(cleanResponse))['links']

[{'type': 'about page',
  'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
 {'type': 'about page', 'url': 'https://edwarddonner.com/'},
 {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
 {'type': 'courses', 'url': 'https://edwarddonner.com/curriculum/'},
 {'type': 'courses', 'url': 'https://edwarddonner.com/'},
 {'type': 'services', 'url': 'https://edwarddonner.com/proficient/'},
 {'type': 'services', 'url': 'https://edwarddonner.com/connect-four/'},
 {'type': 'services', 'url': 'https://edwarddonner.com/outsmart/'},
 {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
 {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'}]